In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support, confusion_matrix
import mlflow

# Reproducibilidad
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

C:\Users\trodr\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


In [2]:
# ---- Config ---------------------------------------------------------------
BASE_DIR = r"C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-completo-merge"
CSV_PATH = os.path.join(BASE_DIR, "df_subset_3000.csv")
IMG_DIR  = os.path.join(BASE_DIR, "imagenes-crop-8-3000")  # <-- cambiar segun condicion

LABEL_COLS = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema",
    "Enlarged Cardiomediastinum", "Fracture", "Lung Lesion", "Lung Opacity",
    "No Finding", "Pleural Effusion", "Pleural Other", "Pneumonia",
    "Pneumothorax",
]
# ---------------------------------------------------------------------------

df = pd.read_csv(CSV_PATH)
print(f"Total imágenes: {len(df)}")

# U-zeros: -1 y NaN → 0
for c in LABEL_COLS:
    df[c] = df[c].fillna(0).replace(-1.0, 0).astype(float)

# Construir full_path apuntando a IMG_DIR
df["full_path"] = df["full_path"].apply(
    lambda p: os.path.join(IMG_DIR, str(p).replace("/", os.sep))
)

# Verificacion
existen = df["full_path"].apply(os.path.exists)
print(f"Imágenes encontradas: {existen.sum()} / {len(df)}")
if existen.sum() < len(df):
    print(f"⚠ Faltan {len(df) - existen.sum()} imágenes — dropeando")
    df = df[existen].reset_index(drop=True)
print(f"Shape final: {df.shape}")

Total imágenes: 3000
Imágenes encontradas: 2990 / 3000
⚠ Faltan 10 imágenes — dropeando
Shape final: (2990, 20)


In [3]:
# Split paciente-nivel (sin data leakage)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, val_idx = next(gss.split(df, groups=df["subject_id"]))

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df   = df.iloc[val_idx].reset_index(drop=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)}")
solapan = set(train_df["subject_id"]) & set(val_df["subject_id"])
print(f"Pacientes solapados (debe ser 0): {len(solapan)}")

Train: 2397 | Val: 593
Pacientes solapados (debe ser 0): 0


In [4]:
IMG_SIZE = 224

train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=5),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class ToraxDataset(Dataset):
    """Dataset con pre-carga en RAM para evitar I/O lento en Windows."""
    def __init__(self, df, label_cols, transform=None):
        self.labels    = torch.tensor(df[label_cols].values, dtype=torch.float32)
        self.transform = transform

        print("Pre-cargando imágenes en RAM...")
        self.images = []
        for path in tqdm(df["full_path"].values):
            img = Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
            self.images.append(img.copy())
        print(f"  {len(self.images)} imágenes en memoria")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = self.images[idx]
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

train_ds = ToraxDataset(train_df, LABEL_COLS, train_tf)
val_ds   = ToraxDataset(val_df,   LABEL_COLS, val_tf)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=0)

imgs, lbls = next(iter(train_loader))
print(f"\nBatch imágenes: {imgs.shape}")
print(f"Batch etiquetas: {lbls.shape}")

Pre-cargando imágenes en RAM...


100%|██████████| 2397/2397 [01:18<00:00, 30.35it/s]


  2397 imágenes en memoria
Pre-cargando imágenes en RAM...


100%|██████████| 593/593 [00:22<00:00, 26.05it/s]


  593 imágenes en memoria

Batch imágenes: torch.Size([32, 3, 224, 224])
Batch etiquetas: torch.Size([32, 13])


In [10]:
# DenseNet-121 preentrenado en ImageNet, cabeza reemplazada para multi-label
model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
in_features = model.classifier.in_features
model.classifier = nn.Linear(in_features, len(LABEL_COLS))
model = model.to(DEVICE)

print(f"Parámetros totales: {sum(p.numel() for p in model.parameters()):,}")
print(f"Parámetros entrenables: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Verificar shapes
x_prueba = torch.randn(2, 3, 224, 224).to(DEVICE)
salida = model(x_prueba)
print(f"Entrada: {x_prueba.shape}")
print(f"Salida:  {salida.shape}")

Parámetros totales: 6,967,181
Parámetros entrenables: 6,967,181
Entrada: torch.Size([2, 3, 224, 224])
Salida:  torch.Size([2, 13])


In [11]:
# pos_weight desde el train set
train_labels = torch.tensor(train_df[LABEL_COLS].fillna(0).values, dtype=torch.float32)
n_pos = (train_labels == 1).sum(dim=0).clamp(min=1)
n_neg = (train_labels == 0).sum(dim=0).clamp(min=1)
pos_weight = (n_neg / n_pos).to(DEVICE) * 0.75

print("pos_weight por label:")
for name, w in zip(LABEL_COLS, pos_weight.cpu()):
    print(f"  {name:<30} {w:.2f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)  # lr mas bajo para fine-tuning
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=3
)

# Verificacion
imgs, lbls = next(iter(train_loader))
imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
logits = model(imgs)
loss = criterion(logits, lbls)
print(f"\nLogits shape: {logits.shape}")
print(f"Labels shape: {lbls.shape}")
print(f"Pérdida inicial: {loss.item():.4f}")

pos_weight por label:
  Atelectasis                    2.99
  Cardiomegaly                   3.06
  Consolidation                  15.59
  Edema                          5.09
  Enlarged Cardiomediastinum     22.90
  Fracture                       43.10
  Lung Lesion                    25.30
  Lung Opacity                   2.69
  No Finding                     1.51
  Pleural Effusion               2.45
  Pleural Other                  149.06
  Pneumonia                      9.46
  Pneumothorax                   15.74

Logits shape: torch.Size([32, 13])
Labels shape: torch.Size([32, 13])
Pérdida inicial: 0.9245


In [12]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    loss_total = 0.0

    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)

        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, lbls)
        loss.backward()
        optimizer.step()

        loss_total += loss.item()

    return loss_total / len(loader)

In [13]:
def evaluate(model, loader, criterion, device, label_cols, umbral=0.5):
    model.eval()
    loss_total = 0.0
    todas_probs, todas_lbls = [], []

    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            logits = model(imgs)
            loss_total += criterion(logits, lbls).item()

            probs = torch.sigmoid(logits)
            todas_probs.append(probs.cpu())
            todas_lbls.append(lbls.cpu())

    y_prob = torch.cat(todas_probs).numpy()
    y_true = torch.cat(todas_lbls).numpy()
    y_true = np.clip(np.nan_to_num(y_true, nan=0.0), 0, 1)
    y_pred = (y_prob >= umbral).astype(int)

    aucs = {}
    for i, nombre in enumerate(label_cols):
        if len(np.unique(y_true[:, i])) < 2:
            continue
        aucs[nombre] = roc_auc_score(y_true[:, i], y_prob[:, i])

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    prec_arr, rec_arr, f1_arr, _ = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )

    confusion_por_etiqueta = {}
    for i, nombre in enumerate(label_cols):
        tn, fp, fn, tp = confusion_matrix(
            y_true[:, i], y_pred[:, i], labels=[0, 1]
        ).ravel()
        confusion_por_etiqueta[nombre] = {
            "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn)
        }

    metricas_por_etiqueta = {}
    for i, nombre in enumerate(label_cols):
        metricas_por_etiqueta[nombre] = {
            "auc":       aucs.get(nombre, float("nan")),
            "precision": float(prec_arr[i]),
            "recall":    float(rec_arr[i]),
            "f1":        float(f1_arr[i]),
            **confusion_por_etiqueta[nombre],
        }

    metricas = {
        "loss":      loss_total / len(loader),
        "auc":       np.mean(list(aucs.values())) if aucs else float("nan"),
        "precision": precision,
        "recall":    recall,
        "f1":        f1,
    }
    return metricas, aucs, metricas_por_etiqueta

In [14]:
mlflow.set_experiment("torax-densenet121")

NUM_EPOCHS = 20
mejor_auc   = 0.0
mejor_epoch = 0
metricas_por_etiqueta_final = {}

with mlflow.start_run(run_name="densenet121_sin_limpieza"):

    mlflow.log_params({
        "modelo":      "DenseNet-121",
        "condicion":   "sin_limpieza",
        "dataset":     "df_subset_3000",
        "n_etiquetas": len(LABEL_COLS),
        "n_train":     len(train_df),
        "n_val":       len(val_df),
        "lr":          1e-4,
        "batch_size":  32,
        "num_epochs":  NUM_EPOCHS,
        "img_size":    IMG_SIZE,
        "umbral":      0.5,
    })

    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        metricas, aucs_detalle, metricas_por_etiqueta = evaluate(
            model, val_loader, criterion, DEVICE, LABEL_COLS
        )

        print(f"Época {epoch:2d}/{NUM_EPOCHS} | "
              f"train_loss: {train_loss:.4f} | "
              f"val_loss: {metricas['loss']:.4f} | "
              f"val_AUC: {metricas['auc']:.4f}")

        mlflow.log_metrics({
            "train_loss":    train_loss,
            "val_loss":      metricas["loss"],
            "val_auc":       metricas["auc"],
            "val_precision": metricas["precision"],
            "val_recall":    metricas["recall"],
            "val_f1":        metricas["f1"],
        }, step=epoch)

        if metricas["auc"] > mejor_auc:
            mejor_auc   = metricas["auc"]
            mejor_epoch = epoch
            metricas_por_etiqueta_final = metricas_por_etiqueta
            torch.save(model.state_dict(), "mejor_modelo_densenet.pth")
            print(f"   ↑ nuevo mejor AUC: {mejor_auc:.4f} (época {mejor_epoch})")

        scheduler.step(metricas["auc"])

    # Loguear métricas finales por etiqueta
    for label, vals in metricas_por_etiqueta_final.items():
        label_key = label.replace(" ", "_")
        mlflow.log_metrics({
            f"{label_key}_auc":       vals["auc"],
            f"{label_key}_precision": vals["precision"],
            f"{label_key}_recall":    vals["recall"],
            f"{label_key}_f1":        vals["f1"],
            f"{label_key}_TP":        vals["TP"],
            f"{label_key}_FP":        vals["FP"],
            f"{label_key}_FN":        vals["FN"],
            f"{label_key}_TN":        vals["TN"],
        })

    mlflow.log_params({"best_val_auc": mejor_auc, "best_epoch": mejor_epoch})

print(f"\nListo. Mejor AUC: {mejor_auc:.4f} en época {mejor_epoch}")

2026/06/23 18:58:46 INFO mlflow.tracking.fluent: Experiment with name 'torax-densenet121' does not exist. Creating a new experiment.
2026/06/23 18:58:46 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    expo

Época  1/20 | train_loss: 1.0070 | val_loss: 1.0432 | val_AUC: 0.6615
   ↑ nuevo mejor AUC: 0.6615 (época 1)
Época  2/20 | train_loss: 0.8812 | val_loss: 1.0465 | val_AUC: 0.6859
   ↑ nuevo mejor AUC: 0.6859 (época 2)
Época  3/20 | train_loss: 0.8039 | val_loss: 1.0612 | val_AUC: 0.6923
   ↑ nuevo mejor AUC: 0.6923 (época 3)
Época  4/20 | train_loss: 0.7471 | val_loss: 1.0813 | val_AUC: 0.6967
   ↑ nuevo mejor AUC: 0.6967 (época 4)
Época  5/20 | train_loss: 0.6924 | val_loss: 1.1245 | val_AUC: 0.7018
   ↑ nuevo mejor AUC: 0.7018 (época 5)
Época  6/20 | train_loss: 0.6276 | val_loss: 1.1452 | val_AUC: 0.7241
   ↑ nuevo mejor AUC: 0.7241 (época 6)
Época  7/20 | train_loss: 0.5652 | val_loss: 1.2637 | val_AUC: 0.7133
Época  8/20 | train_loss: 0.4947 | val_loss: 1.2545 | val_AUC: 0.7145
Época  9/20 | train_loss: 0.4615 | val_loss: 1.5544 | val_AUC: 0.7051
Época 10/20 | train_loss: 0.4199 | val_loss: 1.4962 | val_AUC: 0.6835
Época 11/20 | train_loss: 0.3672 | val_loss: 1.5191 | val_AUC: 0.7

In [15]:
print(f"\n{'='*60}")
print(f"  RESULTADOS FINALES — {mejor_epoch} épocas")
print(f"{'='*60}")
print(f"  Mejor época:  {mejor_epoch}")
print(f"  Mejor AUC:    {mejor_auc:.4f}")
print(f"{'='*60}")
print(f"\n{'Etiqueta':<30} {'AUC':>6} {'Prec':>6} {'Recall':>6} {'F1':>6} {'TP':>4} {'FP':>4} {'FN':>4} {'TN':>4}")
print(f"{'-'*74}")
for label, vals in metricas_por_etiqueta_final.items():
    auc_str = f"{vals['auc']:.4f}" if not np.isnan(vals['auc']) else "   nan"
    print(f"  {label:<28} {auc_str:>6} {vals['precision']:>6.4f} {vals['recall']:>6.4f} "
          f"{vals['f1']:>6.4f} {vals['TP']:>4} {vals['FP']:>4} {vals['FN']:>4} {vals['TN']:>4}")
print(f"{'='*60}\n")


  RESULTADOS FINALES — 6 épocas
  Mejor época:  6
  Mejor AUC:    0.7241

Etiqueta                          AUC   Prec Recall     F1   TP   FP   FN   TN
--------------------------------------------------------------------------
  Atelectasis                  0.7338 0.3472 0.5537 0.4268   67  126   54  346
  Cardiomegaly                 0.7722 0.3457 0.5909 0.4362   65  123   45  360
  Consolidation                0.7452 0.1104 0.5806 0.1856   18  145   13  417
  Edema                        0.8568 0.3377 0.7391 0.4636   51  100   18  424
  Enlarged Cardiomediastinum   0.6143 0.0759 0.2500 0.1165    6   73   18  496
  Fracture                     0.6587 0.0000 0.0000 0.0000    0   29    6  558
  Lung Lesion                  0.7376 0.0536 0.2143 0.0857    3   53   11  526
  Lung Opacity                 0.6978 0.3758 0.4429 0.4066   62  103   78  350
  No Finding                   0.8525 0.6360 0.7876 0.7037  152   87   41  313
  Pleural Effusion             0.8319 0.5075 0.7113 0.5924  

In [16]:
print(f"\n{'='*60}")
print(f"  RESULTADOS FINALES — {mejor_epoch} épocas")
print(f"{'='*60}")
print(f"  Mejor época:  {mejor_epoch}")
print(f"  Mejor AUC:    {mejor_auc:.4f}")
print(f"{'='*60}")
print(f"\n{'Etiqueta':<30} {'AUC':>6} {'Prec':>6} {'Recall':>6} {'F1':>6} {'TP':>4} {'FP':>4} {'FN':>4} {'TN':>4}")
print(f"{'-'*74}")

# Ordenar por AUC descendente
labels_ordenados = sorted(
    metricas_por_etiqueta_final.items(),
    key=lambda x: x[1]["auc"] if not np.isnan(x[1]["auc"]) else -1,
    reverse=True
)

for label, vals in labels_ordenados:
    auc_str = f"{vals['auc']:.4f}" if not np.isnan(vals['auc']) else "   nan"
    print(f"  {label:<28} {auc_str:>6} {vals['precision']:>6.4f} {vals['recall']:>6.4f} "
          f"{vals['f1']:>6.4f} {vals['TP']:>4} {vals['FP']:>4} {vals['FN']:>4} {vals['TN']:>4}")
print(f"{'='*60}\n")


  RESULTADOS FINALES — 6 épocas
  Mejor época:  6
  Mejor AUC:    0.7241

Etiqueta                          AUC   Prec Recall     F1   TP   FP   FN   TN
--------------------------------------------------------------------------
  Edema                        0.8568 0.3377 0.7391 0.4636   51  100   18  424
  No Finding                   0.8525 0.6360 0.7876 0.7037  152   87   41  313
  Pleural Effusion             0.8319 0.5075 0.7113 0.5924  101   98   41  353
  Cardiomegaly                 0.7722 0.3457 0.5909 0.4362   65  123   45  360
  Consolidation                0.7452 0.1104 0.5806 0.1856   18  145   13  417
  Lung Lesion                  0.7376 0.0536 0.2143 0.0857    3   53   11  526
  Atelectasis                  0.7338 0.3472 0.5537 0.4268   67  126   54  346
  Pneumothorax                 0.6993 0.1071 0.3333 0.1622    9   75   18  491
  Lung Opacity                 0.6978 0.3758 0.4429 0.4066   62  103   78  350
  Fracture                     0.6587 0.0000 0.0000 0.0000  